# Session 8 · Case study: churn pipelines and their evaluation

*Author: course team · Licence: CC-BY-4.0*

Practice tasks of blocks 1 and 2 of Session 8 on the IBM Telco customer churn data (7,043 customers, 26.5 % churners):

| Part | Block | Task |
|---|---|---|
| A | 0:00–0:45 | Build a preprocessing pipeline for the Telco data and compare a churn rule, k-NN and logistic regression |
| B | 1:00–1:45 | Evaluate the churn models with confusion matrices and ROC curves and explain which metric fits the business question |

Tasks for you are marked **Task**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
                             classification_report)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # blanks -> NaN
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
print(X.shape, round(y.mean(), 3))
X.head()

## Part A · Pipeline and model comparison

### A1 · Split once, keep the test set locked

All choices below (features, *k*, `C`) are made by cross-validation on the training part. The test part is used once, in Part B.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(len(X_train), len(X_test))

### A2 · The preprocessing step

Numeric columns: median imputation (for the blank `TotalCharges`) and standardisation. Categorical columns: one-hot encoding that ignores unseen categories.

In [ ]:
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
print(pre.fit(X_train).transform(X_train).shape)
print(pre.get_feature_names_out()[:8])

### A3 · The churn rule as a scikit-learn estimator

To compare the hand-made rule with the models under the same cross-validation, we wrap it in a tiny class. It learns nothing in `fit`; `predict_proba` returns 1 for "month-to-month contract and tenure below 12 months", otherwise 0.

In [ ]:
class ChurnRule(ClassifierMixin, BaseEstimator):
    def __init__(self, max_tenure=12):
        self.max_tenure = max_tenure

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        flag = ((X["Contract"] == "Month-to-month") & (X["tenure"] < self.max_tenure)).to_numpy().astype(float)
        return np.column_stack([1 - flag, flag])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

### A4 · Choose *k* for k-NN by cross-validation

In [ ]:
knn_search = GridSearchCV(make_pipeline(pre, KNeighborsClassifier()),
                          {"kneighborsclassifier__n_neighbors": [1, 5, 15, 30, 50, 100, 200]},
                          cv=cv, scoring="roc_auc", return_train_score=True)
knn_search.fit(X_train, y_train)
res = pd.DataFrame(knn_search.cv_results_)[["param_kneighborsclassifier__n_neighbors", "mean_train_score", "mean_test_score"]]
print(res.round(3).to_string(index=False))
print("best:", knn_search.best_params_)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(res.iloc[:, 0], res["mean_train_score"], marker="o", label="training AUC")
ax.plot(res.iloc[:, 0], res["mean_test_score"], marker="o", label="validation AUC")
ax.set_xscale("log"); ax.set_xlabel("number of neighbours k (log scale)"); ax.set_ylabel("ROC AUC")
ax.set_title("k-NN: overfitting for small k, smoothing for large k"); ax.legend()
plt.show()

### A5 · Compare all candidates with the same cross-validation

In [ ]:
models = {
    "majority class": DummyClassifier(strategy="most_frequent"),
    "rule (month-to-month, tenure < 12)": ChurnRule(),
    f"k-NN (k = {knn_search.best_params_['kneighborsclassifier__n_neighbors']})": knn_search.best_estimator_,
    "logistic regression": make_pipeline(pre, LogisticRegression(max_iter=1000)),
}
import warnings
from sklearn.exceptions import UndefinedMetricWarning

# the majority-class model never predicts "churns", so its precision is undefined (reported as 0)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rows = []
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, **{s: r[f"test_{s}"].mean() for s in scoring}})
comparison = pd.DataFrame(rows).set_index("model").round(3)
comparison

**Task A.** (1) Which model is best by accuracy, which by recall, which by ROC AUC? (2) The rule's ROC AUC is computed from 0/1 "probabilities": why is it lower than its accuracy suggests? (3) Change `max_tenure` of the rule (for example 6 or 24) and add a second rule of your own (for example "electronic check and no tech support"). Does any rule beat k-NN?

In [ ]:
# your rule variants here

## Part B · Evaluation on the test set

Now, and only now, we fit the final candidates on all training data and look at the test set once.

In [ ]:
final = {name: m.fit(X_train, y_train) for name, m in models.items() if name != "majority class"}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, m) in zip(axes, final.items()):
    ConfusionMatrixDisplay.from_estimator(m, X_test, y_test, display_labels=["stays", "churns"],
                                          colorbar=False, cmap="Blues", ax=ax)
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
for name, m in final.items():
    print(name)
    print(classification_report(y_test, m.predict(X_test), target_names=["stays", "churns"], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, m in final.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
axes[1].axhline(y_test.mean(), ls="--", color="grey", lw=1, label="random ranking")
axes[0].set_title("ROC curves (test set)"); axes[1].set_title("Precision–recall curves (test set)")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Task B.** Three business questions; for each, choose the metric (or curve) that fits and say which model you would use.

1. *Marketing* has budget for 300 retention calls per quarter and wants as many of the called customers as possible to be real churners.
2. *Management* wants to know whether the model ranks customers by risk better than the current rule, independent of any campaign.
3. *Customer care* wants to miss as few churners as possible, accepting many unnecessary calls.

Write your answers in a Markdown cell below. Theory page 03 then turns question 1 into a cost-based threshold.

*Your answers here.*

**Team project.** Build the baseline and a first validated model for your project: a majority or rule baseline, one model in a pipeline, cross-validated scores with the metric that fits your question.